---
title: Realized volatility measures by session
execute:
  enabled: true
---

`q.indicator.realized_volatility` groups ordered sampled prices by an explicit session label and returns realized variance, quarticity, bipower variation, MedRV, and MinRV for each session. It accepts one instrument per call.

Qrt's offline sample data is daily, so this example uses calendar months as aggregation sessions and daily closes as samples. For production exchange-session estimates, pass intraday sampled prices and actual exchange session labels.

In [1]:
import pandas as pd

import qrt as q

prices = pd.concat(
    {
        "AAPL": q.data.datasets.load("aapl")["close"],
        "SPY": q.data.datasets.load("spy")["close"],
    },
    axis=1,
    join="inner",
).dropna()
end = prices.index.max()
prices = prices.loc[end - pd.DateOffset(years=5) :]
prices.tail()

,AAPL,SPY
datetime,,
2026-07-13,317.309998,749.169983
2026-07-14,314.859985,751.830017
2026-07-15,327.500000,754.809998
2026-07-16,333.260010,750.719971
2026-07-17,333.739990,743.289978


## Calculate the indicator

The input names its observation time, positive sampled price, and session explicitly. Each session needs at least four prices.

In [2]:
def monthly_observations(series: pd.Series) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "time": series.index,
            "price": series.to_numpy(),
            "session": series.index.to_period("M").to_timestamp(),
        }
    )

measures = {
    symbol: q.indicator.realized_volatility(monthly_observations(prices[symbol]))
    for symbol in prices
}
measures["AAPL"].tail()

,realized_variance,realized_quarticity,bipower_variation,med_rv,min_rv
session,,,,,
2026-03-01,0.003049,0.000009,0.002772,0.002734,0.002805
2026-04-01,0.004531,0.000017,0.004979,0.006940,0.005671
2026-05-01,0.002200,0.000005,0.002029,0.001866,0.001786
2026-06-01,0.009816,0.000135,0.008642,0.008157,0.007483
2026-07-01,0.004565,0.000028,0.003352,0.001987,0.002268


## Compare with SPY

The function returns all five measures. The chart selects realized variance for a direct AAPL/SPY comparison; inspect either returned frame to use the other columns.

In [3]:
comparison = pd.DataFrame(
    {
        f"{symbol} realized variance": frame["realized_variance"]
        for symbol, frame in measures.items()
    }
).mul(10_000)
fig = q.plot.col(
    comparison,
    title="AAPL and SPY monthly realized variance from daily samples",
    ylabel="Realized variance (×10,000)",
)
fig.show(renderer="notebook_connected")